In [1]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import datetime as dt

In [ ]:
# Parameters
min_rest_days = 3
max_matches_per_venue_day = 1
min_matches_per_city = 3
max_matches_per_city = 6
max_matches_per_kickoff = 20

# Weights for fan heat exposure indoor and outdoor
# 80% time spent indoors / 20% time spent outdoors
w_in = 0.8
w_out = 0.2

# Plausible dates for matches
matchday_windows = {
    1: (dt.date(2026, 6, 11), dt.date(2026, 6, 16)),
    2: (dt.date(2026, 6, 17), dt.date(2026, 6, 22)),
    3: (dt.date(2026, 6, 23), dt.date(2026, 6, 27)),
}

matches = pd.read_csv("group_positions.csv")
venues = pd.read_csv("stadiums.csv")
wbgt = pd.read_csv("wbgt_data.csv")

matches.head()

,group,match_id,team1,team2,matchday
0,A,A_1,1,2,1
1,A,A_2,3,4,1
2,A,A_3,1,3,2
3,A,A_4,2,4,2
4,A,A_5,1,4,3


In [ ]:
# Generates possible dates within a matchday window
def dates_in_window(matchday):
    start, end = matchday_windows[matchday]
    n_days = (end - start).days + 1
    
    return [
        start + dt.timedelta(days=i)
        for i in range(n_days)
    ]

# Stores all possible schedule options for each matchday
options_by_matchday = {}
option_id = 0

for md in matchday_windows:
    options = []
    # All possible dates within this matchday window
    for d in dates_in_window(md):
        # All possible stadiums
        for _, venue in venues.iterrows():
            # Get WBGT values for this venue's city
            city_wbgt = wbgt[wbgt.city == venue.city]
            # Each kickoff time creates a possible schedule option
            for _, weather in city_wbgt.iterrows():
                # Weighted fan heat exposure risk
                risk = round(
                    w_in * weather.wbgt_indoor +
                    w_out * weather.wbgt_outdoor,
                    2
                )
                options.append({
                    "option_id": option_id,
                    "matchday": md,
                    "city": venue.city,
                    "stadium": venue.stadium_name,
                    "capacity": venue.capacity,
                    "date": d,
                    "date_idx": (d - dt.date(2026,1,1)).days,
                    "kickoff_time": weather.kickoff_time,
                    "risk": risk,
                })
                option_id += 1
    options_by_matchday[md] = pd.DataFrame(options)

# Combine all schedule options
schedule_options = (
    pd.concat(options_by_matchday.values(), ignore_index=True)
    .set_index("option_id")
)

print(f"Total schedule options = {len(schedule_options)}")

schedule_options.head()

Total schedule options = 1088


,matchday,city,stadium,capacity,date,date_idx,kickoff_time,risk
option_id,,,,,,,,
0,1,Kansas City,Arrowhead Stadium,69045,2026-06-11,161,12:00,28.93
1,1,Kansas City,Arrowhead Stadium,69045,2026-06-11,161,15:00,29.85
2,1,Kansas City,Arrowhead Stadium,69045,2026-06-11,161,18:00,28.42
3,1,Kansas City,Arrowhead Stadium,69045,2026-06-11,161,21:00,25.07
4,1,Toronto,BMO Field,43036,2026-06-11,161,12:00,23.93


In [11]:
m = gp.Model("wbgt_heat_risk_scheduling")

# Decision variable - x[match_id, option_id]
x = {}

# Stores which schedule options are available for each match
options_for_match = {}

for _, match in matches.iterrows():
    # A match can only use schedule options from its assigned matchday
    valid_options = list(
        schedule_options[
            schedule_options.matchday == match.matchday
        ].index
    )

    options_for_match[match.match_id] = valid_options

    for option_id in valid_options:
        x[match.match_id, option_id] = m.addVar(
            vtype=GRB.BINARY,
            name=f"x_{match.match_id}_{option_id}"
        )

m.update()

print(f"Decision variables: {len(x)}")

Decision variables: 26112


In [12]:
# Minimize heat-risk exposure per person
obj = gp.quicksum(
    schedule_options.loc[option_id, "risk"] * schedule_options.loc[option_id, "capacity"]* x[match_id, option_id]

    for match_id, options in options_for_match.items()
    for option_id in options
)

m.setObjective(obj, GRB.MINIMIZE)

In [13]:
# Constraint - every match scheduled exactly once
for match_id, options in options_for_match.items():
    m.addConstr(gp.quicksum(x[match_id, option_id] for option_id in options) == 1, name=f"assign_{match_id}")

# A single stadium/date/kickoff option can only be used once
for option_id in schedule_options.index:
    users = [
        x[match_id, option_id]
        for match_id, options in options_for_match.items()
        if option_id in options
    ]
    if users:
        m.addConstr(gp.quicksum(users) <= 1, name=f"option_once_{option_id}")

# Maximum matches at one stadium per day
for (stadium, date), idx in schedule_options.groupby(
    ["stadium", "date"]
).groups.items():
    users = [
        x[match_id, option_id]
        for match_id, options in options_for_match.items()
        for option_id in options
        if option_id in idx
    ]

    if users:
        m.addConstr(gp.quicksum(users) <= max_matches_per_venue_day, name=f"stadiumcap_{stadium}_{date}")

# Max and min matches per city
for city in venues.city.unique():
    users = [
        x[match_id, option_id]
        for match_id, options in options_for_match.items()
        for option_id in options
        if schedule_options.loc[option_id,"city"] == city
    ]
    if users:
        m.addConstr(
            gp.quicksum(users) >= min_matches_per_city,
            name=f"mincity_{city}"
        )
        m.addConstr(
            gp.quicksum(users) <= max_matches_per_city,
            name=f"maxcity_{city}"
        )

# Creates numerical date value to be used in rest day constraints
def date_index_expr(match_id):
    expr = gp.LinExpr()
    for option_id in options_for_match[match_id]:
        expr += (
            schedule_options.loc[option_id,"date_idx"]
        ) * x[match_id, option_id]
    return expr

# Rest days between team matches
for group, grp_rows in matches.groupby("group"):
    for team in [1,2,3,4]:
        seq = []
        for md in [1,2,3]:
            md_rows = grp_rows[
                grp_rows.matchday == md
            ]
            hit = md_rows[
                (md_rows.team1 == team)
                |
                (md_rows.team2 == team)
            ]
            if len(hit) == 1:
                seq.append(
                    hit.iloc[0].match_id
                )
        for a,b in zip(seq, seq[1:]):
            m.addConstr(date_index_expr(b) - date_index_expr(a) >= min_rest_days, name=f"rest_{group}_{team}_{a}_{b}")

# Limit number of matches at each kickoff time
for kickoff in schedule_options.kickoff_time.unique():
    users = [
        x[match_id, option_id]
        for match_id, options in options_for_match.items()
        for option_id in options
        if schedule_options.loc[option_id, "kickoff_time"] == kickoff
    ]

    m.addConstr(
        gp.quicksum(users) <= max_matches_per_kickoff,
        name=f"kickoff_limit_{kickoff}"
    )

m.update()

print(f"Total constraints: {m.NumConstrs}")

Total constraints: 1564


In [14]:
m.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i5-1335U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 1564 rows, 26112 columns and 227328 nonzeros (Min)
Model fingerprint: 0x71523dbd
Model has 26112 linear objective coefficients
Variable types: 0 continuous, 26112 integer (26112 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [9e+05, 2e+06]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]

Found heuristic solution: objective 1.114454e+08
Presolve added 0 rows and 96 columns
Presolve removed 992 rows and 0 columns
Presolve time: 0.20s
Presolved: 572 rows, 26208 columns, 150816 nonzeros
Variable types: 0 continuous, 26208 integer (26112 binary)

Root relaxation: objective 9.956635e+07, 625 iterations, 0.04 seconds (0.08 work units)

    Nodes    |    Current Node    |    

In [ ]:
results = []

if m.SolCount > 0:
    for match_id, options in options_for_match.items():
        for option_id in options:
            if x[match_id, option_id].X > 0.5:
                option = schedule_options.loc[option_id]
                match = matches[
                    matches.match_id == match_id
                ].iloc[0]

                results.append({
                    "match_id": match_id,
                    "group": match.group,
                    "team1": match.team1,
                    "team2": match.team2,
                    "matchday": match.matchday,

                    "city": option.city,
                    "stadium": option.stadium,
                    "date": option.date,
                    "kickoff_time": option.kickoff_time,

                    "risk": option.risk,
                    "capacity": option.capacity,

                    "person_weighted_risk":
                        option.risk * option.capacity
                })

    results_df = (
        pd.DataFrame(results)
        .sort_values(["group","matchday"])
    )

    print(f"Total person-weighted risk: {m.ObjVal:,.0f}")
    print(f"Average WBGT per match: {results_df.risk.mean():.2f}")
    results_df

else:

    print("No feasible solution found.")

Total person-weighted risk: 99,566,348
Average WBGT per match: 22.23


In [ ]:
# Save optimized schedule
results_df.to_csv(
    "optimized_schedule_latest.csv",
    index=False
)

In [17]:
# Load official FIFA schedule
real_schedule = pd.read_csv("real_schedule.csv")

kick_off_times = [12, 15, 18, 21]

# Round kickoff times to match available WBGT data
def nearest_kickoff(t):
    hour = int(t.split(":")[0])
    return f"{min(kick_off_times, key=lambda c: abs(c-hour)):02d}:00"

real_schedule["kickoff_rounded"] = (
    real_schedule.kickoff_time_local
    .apply(nearest_kickoff)
)

# Attach WBGT + stadium capacity
real_scored = (
    real_schedule

    .merge(
        wbgt,

        left_on=[
            "city",
            "kickoff_rounded"
        ],

        right_on=[
            "city",
            "kickoff_time"
        ],

        how="left"
    )

    .merge(
        venues[["city","capacity"]],

        on="city",

        how="left"
    )
)

# Calculate weighted fan heat risk
real_scored["risk"] = (
    w_in * real_scored.wbgt_indoor + w_out * real_scored.wbgt_outdoor).round(2)

# Same objective calculation as optimizer
real_scored["person_weighted_risk"] = (
    real_scored.risk
    *
    real_scored.capacity
)


print(f"Total person-weighted risk: " f"{real_scored.person_weighted_risk.sum():,.0f}")
print(f"Average WBGT per match: " f"{real_scored.risk.mean():.2f}")

Total person-weighted risk: 108,196,807
Average WBGT per match: 23.21
